# Relatorio de Resultados do TCC

Gera, em uma execucao, todas as tabelas e figuras da secao **Resultados e Discussao**
a partir dos artefatos que o pipeline `hypertension-mlops` ja persistiu.

## O que este notebook e

Camada de **relatorio**, nao de execucao. Ele nao treina, nao transforma e nao
reimplementa nada de `src/`. Le os arquivos abaixo e renderiza as saidas.

| Saida | Arquivo de origem |
|---|---|
| Tabela 10, desempenho por execucao | `artifacts/<run>/metrics.json` |
| Tabela de coeficientes, eq. (8) | `artifacts/lr_production/interpretation.json` |
| Figura 8, curvas ROC | `artifacts/<run>/roc.json` |
| Figura 9, matriz de confusao | `artifacts/lr_production/metrics.json` |
| Tabela 11, regra clinica | `evidencias/rule_baseline.json` |
| Tabela 12, subgrupos | `evidencias/subgroup_metrics_lr_production.csv` |
| Figura 10, importancia | `evidencias/global_importance.json` |
| Tabela 13, fidelidade | `evidencias/golden_metrics.json` e `fase0_reexecucao_metricas.json` |
| Tabela 14, carga e eq. (9) | `evidencias/load_test_*.json` e `configs/config.yaml` |
| Tabela 15, desvio de dados | `evidencias/drift_simulated.json` e `drift_baseline.json` |
| Tabela 16, objetivos especificos | todos os anteriores, mais `coverage_local_trace.txt` |
| Registro do modelo | `artifacts/lr_production/registration.json` |

Consequencia direta: se um numero divergir do pipeline, o bug e do pipeline ou do
artefato, nunca deste notebook. E, como tudo sai de uma unica leitura, as
divergencias internas que o texto registra hoje nao podem reaparecer.

## O que ele nao produz

As Figuras 11, 12 e 13 sao capturas de tela de servicos no ar e a quarta linha da
Tabela 14 exige a API em contentor. As duas coisas precisam de `docker compose up`.
Os comandos exatos saem no relatorio de pendencias, ao final.

## Onde executar

Da raiz do repositorio, no ambiente de treino:

```
docker compose -f docker/docker-compose.yml --profile tools run --rm train python scripts/run_relatorio.py --html
```

## Regra de degradacao

Nenhuma celula interrompe a execucao. Secao sem artefato registra `[PENDENTE]` com o
motivo e o comando que a resolve, e a execucao continua.

In [ ]:
from __future__ import annotations

import hashlib
import json
import subprocess
import sys
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Iterable

import numpy as np
import pandas as pd
import yaml

# Tabelas largas e textos longos precisam aparecer inteiros na celula.
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 140)


def localizar_raiz(inicio: Path | None = None) -> Path:
    """Localiza a raiz do repositorio subindo a arvore de diretorios.

    Parameters
    ----------
    inicio : Path or None, optional
        Diretorio de partida. Quando ausente, usa o diretorio corrente.

    Returns
    -------
    Path
        Primeiro ancestral que contenha `configs/config.yaml` e `src/`.

    Raises
    ------
    FileNotFoundError
        Quando nenhum ancestral satisfaz o criterio.
    """
    atual = (inicio or Path.cwd()).resolve()
    for candidato in [atual, *atual.parents]:
        if (candidato / "configs" / "config.yaml").is_file() and (candidato / "src").is_dir():
            return candidato
    raise FileNotFoundError(f"Raiz do repositorio nao encontrada a partir de {atual}.")


RAIZ: Path = localizar_raiz()
ARTEFATOS: Path = RAIZ / "artifacts"
EVIDENCIAS: Path = RAIZ / "evidencias"
SAIDA: Path = EVIDENCIAS / "relatorio_resultados"
(SAIDA / "figuras").mkdir(parents=True, exist_ok=True)
(SAIDA / "tabelas").mkdir(parents=True, exist_ok=True)

# Consumo somente leitura do mesmo arquivo que o pipeline usa. Nenhum limiar,
# nome de execucao ou papel e repetido aqui: tudo vem da configuracao.
CONFIG: dict[str, Any] = yaml.safe_load(
    (RAIZ / "configs" / "config.yaml").read_text(encoding="utf-8")
)

SEED: int = CONFIG["project"]["seed"]
N_FOLDS: int = CONFIG["split"]["n_splits"]
THRESHOLD: float = CONFIG["model"]["decision_threshold"]
RUN_PRODUCAO: str = CONFIG["tracking"]["promote_run"]
RUN_FIDELIDADE: str = next(
    r["name"] for r in CONFIG["model"]["runs"] if r["role"] == "fidelity"
)
ACEITACAO: dict[str, Any] = CONFIG["acceptance"]
FIDELIDADE: dict[str, Any] = CONFIG["fidelity"]

# Papel em portugues, para as legendas do TCC.
PAPEL_PT: dict[str, str] = {
    "fidelity": "fidelidade",
    "champion": "campeao",
    "comparison": "comparacao",
    "baseline": "sem ponderacao",
    "candidate": "candidato",
}
ORDEM_RUNS: list[tuple[str, str]] = [
    (r["name"], PAPEL_PT.get(r["role"], r["role"])) for r in CONFIG["model"]["runs"]
]
# A ordem do TCC e por papel e depois por desempenho; a ordenacao final e feita
# na montagem da Tabela 10, sobre os valores lidos.

TOPO_IMPORTANCIA: int = 12

print(f"Raiz          : {RAIZ}")
print(f"Saida         : {SAIDA}")
print(f"Execucao promovida: {RUN_PRODUCAO} | fidelidade: {RUN_FIDELIDADE}")
print(f"Seed {SEED}, k={N_FOLDS}, limiar {THRESHOLD}")
print(f"{len(ORDEM_RUNS)} execucoes declaradas em configuracao")

## 1. Utilitarios e proveniencia

Cada tabela passa por uma funcao unica de registro, que grava CSV com precisao
completa para auditoria e Markdown em formato brasileiro para colar no TCC. Cada
figura sai sem titulo e sem borda, em 300 pontos por polegada, conforme o item 15.1
do Manual.

A proveniencia registra o ambiente, o commit e os resumos criptograficos que
produziram os numeros, e responde as duas marcas `[PROVENIENCIA]` da lista de
fechamento do texto.

In [ ]:
PENDENCIAS: list[dict[str, str]] = []
TABELAS: dict[str, pd.DataFrame] = {}
VALORES: dict[str, Any] = {}   # valores calculados, consumidos pela Tabela 16


def registrar_pendencia(item: str, motivo: str, acao: str = "") -> None:
    """Registra um item que nao pode ser produzido nesta execucao."""
    PENDENCIAS.append({"item": item, "motivo": motivo, "acao": acao})
    print(f"[PENDENTE] {item}: {motivo}")
    if acao:
        print(f"           acao: {acao}")


def secao(item: str) -> Callable:
    """Decorador que converte excecao de uma secao em pendencia registrada.

    Parameters
    ----------
    item : str
        Identificacao do artefato do TCC, por exemplo "Tabela 12".

    Returns
    -------
    Callable
        Decorador que devolve None quando a funcao falha.
    """
    def envoltorio(funcao: Callable) -> Callable:
        def executar(*args: Any, **kwargs: Any) -> Any:
            try:
                return funcao(*args, **kwargs)
            except Exception as erro:  # registro deliberado, nunca silencioso
                registrar_pendencia(item, f"{type(erro).__name__}: {erro}",
                                    "Ver o traceback impresso abaixo.")
                traceback.print_exc(limit=4)
                return None
        return executar
    return envoltorio


def ler_json(caminho: Path) -> Any:
    """Le um arquivo JSON do repositorio.

    Raises
    ------
    FileNotFoundError
        Quando o artefato nao existe, com o caminho relativo na mensagem.
    """
    if not caminho.is_file():
        raise FileNotFoundError(f"artefato ausente: {caminho.relative_to(RAIZ)}")
    return json.loads(caminho.read_text(encoding="utf-8"))


def fmt_br(valor: Any, casas: int = 4) -> str:
    """Formata um numero no padrao brasileiro, com virgula decimal."""
    if valor is None or (isinstance(valor, float) and not np.isfinite(valor)):
        return "[PEND.]"
    if isinstance(valor, (bool, np.bool_)):
        return "sim" if valor else "nao"
    if isinstance(valor, (int, np.integer)):
        return f"{valor:,}".replace(",", ".")
    if isinstance(valor, (float, np.floating)):
        return f"{valor:.{casas}f}".replace(".", ",")
    return str(valor)


def quadro_para_markdown(dados: pd.DataFrame) -> str:
    """Converte um DataFrame em tabela Markdown sem depender de `tabulate`.

    `DataFrame.to_markdown` exige a biblioteca `tabulate`, ausente da imagem de
    treino. Acrescenta-la mudaria o lock que sustenta a referencia de fidelidade.
    """
    def limpar(valor: Any) -> str:
        return str(valor).replace("|", "\\|").replace("\n", " ")

    colunas = [limpar(c) for c in dados.columns]
    corpo = [[limpar(v) for v in linha] for linha in dados.itertuples(index=False)]
    larguras = [max([len(colunas[i])] + [l[i].__len__() for l in corpo])
                for i in range(len(colunas))]

    def montar(celulas: list[str]) -> str:
        return "| " + " | ".join(c.ljust(larguras[i]) for i, c in enumerate(celulas)) + " |"

    return "\n".join([montar(colunas),
                      "|" + "|".join("-" * (w + 2) for w in larguras) + "|",
                      *[montar(linha) for linha in corpo]])


def registrar_tabela(nome: str, dados: pd.DataFrame, titulo: str,
                     casas: int | dict[str, int] = 4, nota: str = "") -> pd.DataFrame:
    """Persiste uma tabela em CSV de auditoria e em Markdown para o TCC.

    Parameters
    ----------
    nome : str
        Identificador do arquivo, por exemplo "tabela_10".
    dados : pandas.DataFrame
        Conteudo sem arredondamento.
    titulo : str
        Legenda no padrao do TCC.
    casas : int or dict, optional
        Casas decimais na versao Markdown, global ou por coluna.
    nota : str, optional
        Nota de rodape.

    Returns
    -------
    pandas.DataFrame
        O proprio conteudo recebido, para encadeamento.
    """
    TABELAS[nome] = dados
    dados.to_csv(SAIDA / "tabelas" / f"{nome}.csv", index=False, encoding="utf-8")

    formatado = dados.copy()
    for coluna in formatado.columns:
        n = casas[coluna] if isinstance(casas, dict) and coluna in casas else (
            casas if isinstance(casas, int) else 4)
        formatado[coluna] = formatado[coluna].map(lambda v, n=n: fmt_br(v, n))

    linhas = [titulo, "", quadro_para_markdown(formatado), "",
              "Fonte: Resultados originais da pesquisa"]
    if nota:
        linhas += ["", f"Nota: {nota}"]
    (SAIDA / "tabelas" / f"{nome}.md").write_text("\n".join(linhas) + "\n", encoding="utf-8")
    print(f"gravado: tabelas/{nome}.csv e {nome}.md")
    display(formatado)
    return dados


def registrar_figura(fig, nome: str, legenda: str) -> None:
    """Persiste e exibe uma figura sem titulo e sem borda (item 15.1 do Manual)."""
    for eixo in fig.get_axes():
        eixo.set_title("")
        for lado in ("top", "right"):
            eixo.spines[lado].set_visible(False)
    fig.savefig(SAIDA / "figuras" / f"{nome}.png", dpi=300, bbox_inches="tight",
                facecolor="white", edgecolor="none")
    (SAIDA / "figuras" / f"{nome}.txt").write_text(legenda, encoding="utf-8")
    print(f"gravado: figuras/{nome}.png")
    display(fig)
    print(legenda)
    print("Fonte: Resultados originais da pesquisa")

In [ ]:
def sha256_arquivo(caminho: Path, bloco: int = 1 << 20) -> str:
    """Calcula o resumo SHA-256 de um arquivo em modo incremental."""
    digestor = hashlib.sha256()
    with caminho.open("rb") as handle:
        for pedaco in iter(lambda: handle.read(bloco), b""):
            digestor.update(pedaco)
    return digestor.hexdigest()


def comando(args: list[str]) -> str:
    """Executa um comando na raiz e devolve a saida, ou um marcador em caso de falha."""
    try:
        return subprocess.run(args, cwd=RAIZ, capture_output=True, text=True,
                              timeout=30, check=True).stdout.strip()
    except (subprocess.SubprocessError, FileNotFoundError, OSError):
        return "[INDISPONIVEL]"


def versoes() -> dict[str, str]:
    """Versoes das bibliotecas que afetam a reprodutibilidade numerica."""
    from importlib.metadata import PackageNotFoundError, version
    alvos = ("numpy", "pandas", "scikit-learn", "scipy", "joblib",
             "xgboost", "lightgbm", "catboost", "mlflow", "matplotlib")
    saida = {}
    for nome in alvos:
        try:
            saida[nome] = version(nome)
        except PackageNotFoundError:
            saida[nome] = "[AUSENTE]"
    return saida


GOLDEN: dict[str, Any] = {}
try:
    GOLDEN = ler_json(EVIDENCIAS / "golden_metrics.json")
except FileNotFoundError as erro:
    print(f"[PENDENTE] {erro}")

REGISTRO: dict[str, Any] = {}
try:
    REGISTRO = ler_json(ARTEFATOS / RUN_PRODUCAO / "registration.json")
except FileNotFoundError as erro:
    print(f"[PENDENTE] {erro}")

PROVENIENCIA: dict[str, Any] = {
    "gerado_em_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "python": sys.version.split()[0],
    "em_container": Path("/.dockerenv").exists(),
    "git_commit": comando(["git", "rev-parse", "HEAD"]),
    "git_tag": comando(["git", "describe", "--tags", "--always"]),
    "git_limpo": comando(["git", "status", "--porcelain"]) == "",
    "seed": SEED,
    "n_folds": N_FOLDS,
    "threshold": THRESHOLD,
    "versoes_relatorio": versoes(),
    "ambiente_do_golden": GOLDEN.get("environment", {}),
    "code_revision_do_golden": GOLDEN.get("code_revision", "[AUSENTE]"),
    "raw_dataset_sha256": GOLDEN.get("raw_dataset_sha256", "[AUSENTE]"),
    "development_frame_sha256": GOLDEN.get("development_frame_sha256", "[AUSENTE]"),
    "registro_do_modelo": REGISTRO,
    "config_sha256": sha256_arquivo(RAIZ / "configs" / "config.yaml"),
}


def commit_curto() -> str:
    """Commit abreviado, preservando o marcador quando indisponivel."""
    c = PROVENIENCIA["git_commit"]
    return c[:8] if c != "[INDISPONIVEL]" else c


(SAIDA / "proveniencia.json").write_text(
    json.dumps(PROVENIENCIA, indent=2, ensure_ascii=False), encoding="utf-8")

# O ambiente que gerou os artefatos precisa ser o mesmo que gera o relatorio,
# senao as versoes registradas no texto nao correspondem ao que produziu os numeros.
divergentes = [
    f"{k}: golden {v} contra relatorio {PROVENIENCIA['versoes_relatorio'].get(k, '?')}"
    for k, v in PROVENIENCIA["ambiente_do_golden"].items()
    if k != "python" and PROVENIENCIA["versoes_relatorio"].get(k) != v
]
if divergentes:
    print("[VERSAO] Ambiente do relatorio difere do que congelou os golden values:")
    for linha in divergentes:
        print(f"  {linha}")
if not PROVENIENCIA["git_limpo"]:
    print("[PROVENIENCIA] Arvore de trabalho suja: os numeros nao correspondem a um "
          "commit identificavel. Faca commit antes de gerar a entrega.")

print(json.dumps(PROVENIENCIA, indent=2, ensure_ascii=False))

## 2. Tabela 10, desempenho por execucao

Fonte: `artifacts/<run>/metrics.json`, um por execucao, gravado pelo proprio runner
de experimentos sem arredondamento. A ordenacao segue a do texto: fidelidade,
campeao, comparacao, baseline, e depois os candidatos por AUC decrescente.

In [ ]:
METRICAS: dict[str, dict[str, Any]] = {}
for nome, _papel in ORDEM_RUNS:
    caminho = ARTEFATOS / nome / "metrics.json"
    if caminho.is_file():
        METRICAS[nome] = ler_json(caminho)

faltando_run = [n for n, _ in ORDEM_RUNS if n not in METRICAS]
print(f"{len(METRICAS)} de {len(ORDEM_RUNS)} execucoes com metrics.json")
if faltando_run:
    registrar_pendencia(
        "Tabela 10", f"execucoes sem metrics.json: {', '.join(faltando_run)}",
        "python -m src.models.train --all   (no ambiente de treino)",
    )


def cv(nome: str, chave: str) -> float:
    """Le uma metrica do resumo de validacao cruzada de uma execucao."""
    return float(METRICAS[nome]["cv"]["summary"][chave]) if nome in METRICAS else np.nan


def holdout(nome: str, chave: str) -> float:
    """Le uma metrica da particao de teste de uma execucao."""
    return float(METRICAS[nome]["holdout"][chave]) if nome in METRICAS else np.nan


@secao("Tabela 10")
def montar_tabela_10() -> pd.DataFrame:
    """Monta a Tabela 10 a partir dos arquivos de metricas de cada execucao."""
    linhas = [{
        "Execucao": nome,
        "Papel": papel,
        "AUC VC": cv(nome, "auc_roc_mean"),
        "Sens. VC": cv(nome, "sensitivity_mean"),
        "Espec. VC": cv(nome, "specificity_mean"),
        "AUC teste": holdout(nome, "auc_roc"),
        "Sens. teste": holdout(nome, "sensitivity"),
    } for nome, papel in ORDEM_RUNS]

    quadro = pd.DataFrame(linhas)
    # Papeis nao candidatos mantem a ordem da configuracao; candidatos por AUC.
    prioridade = {p: i for i, p in enumerate(
        ["fidelidade", "campeao", "comparacao", "sem ponderacao"])}
    quadro["_ordem"] = quadro["Papel"].map(lambda p: prioridade.get(p, 99))
    quadro = quadro.sort_values(["_ordem", "AUC VC"], ascending=[True, False])
    return quadro.drop(columns="_ordem").reset_index(drop=True)


tabela_10 = montar_tabela_10()
if tabela_10 is not None:
    registrar_tabela(
        "tabela_10", tabela_10,
        "Tabela 10. Desempenho por execucao na validacao cruzada e na particao de teste",
        nota="validacao cruzada [VC]",
    )
    VALORES["n_execucoes"] = int(tabela_10["AUC VC"].notna().sum())
    VALORES["auc_cv_producao"] = cv(RUN_PRODUCAO, "auc_roc_mean")
    VALORES["sens_cv_producao"] = cv(RUN_PRODUCAO, "sensitivity_mean")

    melhor_candidato = tabela_10[tabela_10["Papel"] == "candidato"].iloc[0]
    print(f"\nMelhor candidato: {melhor_candidato['Execucao']}, "
          f"AUC {fmt_br(melhor_candidato['AUC VC'])}. "
          f"Vantagem do campeao: "
          f"{fmt_br(VALORES['auc_cv_producao'] - melhor_candidato['AUC VC'])} em AUC e "
          f"{fmt_br(VALORES['sens_cv_producao'] - melhor_candidato['Sens. VC'])} "
          "em sensibilidade.")

## 3. Tabela 10 estendida, com dispersao entre dobras

A Tabela 10 do texto reporta a media. A versao estendida abaixo acrescenta o desvio
padrao entre as dez dobras e as demais metricas, e serve para o Apendice ou para
qualquer afirmacao do texto que cite intervalo.

In [ ]:
@secao("Tabela 10 estendida")
def montar_tabela_10_estendida() -> pd.DataFrame:
    """Versao completa por execucao, com dispersao entre dobras."""
    linhas = [{
        "Execucao": nome,
        "Papel": papel,
        "AUC VC": cv(nome, "auc_roc_mean"),
        "AUC VC dp": cv(nome, "auc_roc_std"),
        "Sens. VC": cv(nome, "sensitivity_mean"),
        "Sens. VC dp": cv(nome, "sensitivity_std"),
        "Espec. VC": cv(nome, "specificity_mean"),
        "Prec. VC": cv(nome, "precision_mean"),
        "F1 VC": cv(nome, "f1_mean"),
        "AUC teste": holdout(nome, "auc_roc"),
        "Sens. teste": holdout(nome, "sensitivity"),
        "Espec. teste": holdout(nome, "specificity"),
        "Prec. teste": holdout(nome, "precision"),
        "F1 teste": holdout(nome, "f1"),
    } for nome, papel in ORDEM_RUNS if nome in METRICAS]
    return pd.DataFrame(linhas).sort_values("AUC VC", ascending=False).reset_index(drop=True)


tabela_10e = montar_tabela_10_estendida()
if tabela_10e is not None:
    registrar_tabela(
        "tabela_10_estendida", tabela_10e,
        "Tabela 10 estendida. Todas as metricas por execucao, com dispersao entre dobras",
        nota="dp indica desvio padrao entre as dez dobras",
    )

## 4. Coeficientes do modelo promovido, eq. (8)

Fonte: `artifacts/lr_production/interpretation.json`. O texto compara dois
coeficientes com os do estudo de origem, que foram estimados sobre colunas
escalonadas; aqui saem na escala das colunas binarias, sem escalonamento, como o
modelo servido de fato opera.

In [ ]:
@secao("Tabela de coeficientes")
def montar_coeficientes(topo: int = 20) -> pd.DataFrame:
    """Coeficientes da regressao promovida, ordenados por contribuicao media."""
    dados = ler_json(ARTEFATOS / RUN_PRODUCAO / "interpretation.json")
    VALORES["intercepto"] = float(dados["intercept"])
    VALORES["n_train"] = int(dados["n_train"])
    VALORES["n_dummies"] = len(dados["coefficients"])
    quadro = pd.DataFrame(dados["coefficients"])
    quadro = quadro.rename(columns={
        "dummy": "Coluna binaria", "label_pt": "Variavel", "category": "Categoria",
        "coefficient": "Coeficiente", "mean_abs_contribution": "Contrib. media |.|",
    })[["Coluna binaria", "Variavel", "Categoria", "Coeficiente", "Contrib. media |.|"]]

    print(f"Intercepto: {fmt_br(VALORES['intercepto'])}")
    print(f"Colunas binarias na matriz de projeto: {VALORES['n_dummies']}")
    print(f"Observacoes no treino: {VALORES['n_train']}")
    # Os dois coeficientes citados nominalmente no texto.
    for alvo in ("sex_2", "age_group_60-69"):
        linha = quadro[quadro["Coluna binaria"] == alvo]
        if not linha.empty:
            valor = float(linha["Coeficiente"].iloc[0])
            VALORES[f"coef_{alvo}"] = valor
            print(f"  {alvo}: {fmt_br(valor)}")
    return quadro.reindex(
        quadro["Contrib. media |.|"].abs().sort_values(ascending=False).index
    ).head(topo).reset_index(drop=True)


coeficientes = montar_coeficientes()
if coeficientes is not None:
    registrar_tabela(
        "tabela_coeficientes", coeficientes,
        "Tabela de apoio. Vinte colunas binarias de maior contribuicao media na "
        "execucao de producao",
        nota="coeficientes na escala de logito, sobre colunas binarias nao escalonadas",
    )

## 5. Figura 8, curvas ROC na particao de teste

Fonte: `artifacts/<run>/roc.json`, bloco `holdout`, com a AUC lida do mesmo
`metrics.json` que alimenta a Tabela 10. Sem titulo e sem borda, conforme o item
15.1 do Manual.

In [ ]:
import matplotlib.pyplot as plt

# Sem `matplotlib.use("Agg")`: o backend padrao do Jupyter renderiza na celula, e
# em execucao headless o backend ja e nao interativo.
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9,
                     "axes.grid": True, "grid.alpha": 0.25, "axes.axisbelow": True})


@secao("Figura 8")
def desenhar_figura_8() -> None:
    """Curvas ROC na particao de teste, uma por execucao."""
    curvas: list[tuple[str, dict[str, list[float]], float]] = []
    ausentes: list[str] = []
    for nome, _papel in ORDEM_RUNS:
        caminho = ARTEFATOS / nome / "roc.json"
        if not caminho.is_file() or nome not in METRICAS:
            ausentes.append(nome)
            continue
        curvas.append((nome, ler_json(caminho)["holdout"], holdout(nome, "auc_roc")))
    if not curvas:
        raise FileNotFoundError("nenhum roc.json encontrado")

    curvas.sort(key=lambda t: t[2], reverse=True)
    fig, eixo = plt.subplots(figsize=(5.4, 4.6))
    for nome, pontos, auc in curvas:
        destaque = nome == RUN_PRODUCAO
        eixo.plot(pontos["fpr"], pontos["tpr"], lw=2.2 if destaque else 1.0,
                  zorder=3 if destaque else 2,
                  label=f"{nome} (AUC {fmt_br(auc, 4)})")
    eixo.plot([0, 1], [0, 1], ls="--", lw=0.8, color="0.55", zorder=1,
              label="classificador aleatorio")
    # Nomenclatura dos eixos conforme a figura da dissertacao de origem, para que a
    # comparacao visual com ela nao exija traducao mental do leitor.
    eixo.set_xlabel("Taxa de Falsos Positivos")
    eixo.set_ylabel("Taxa de Verdadeiros Positivos")
    eixo.set_xlim(0, 1)
    eixo.set_ylim(0, 1.02)
    eixo.legend(loc="lower right", frameon=False, fontsize=7)
    fig.tight_layout()

    n_teste = int(METRICAS[RUN_PRODUCAO]["holdout"]["n"])
    VALORES["n_teste"] = n_teste
    registrar_figura(
        fig, "figura_08_roc",
        f"Figura 8. Curvas ROC na particao de teste de {n_teste:,} registros, por "
        "execucao, com a AUC-ROC indicada na legenda".replace(",", "."),
    )
    plt.close(fig)
    if ausentes:
        registrar_pendencia("Figura 8", f"execucoes sem curva: {', '.join(ausentes)}",
                            "python -m src.models.train --all")


desenhar_figura_8()

## 6. Figura 9, matriz de confusao da execucao de producao

Fonte: `artifacts/lr_production/metrics.json`, bloco `holdout`. As quatro celulas
e todas as metricas derivadas saem do mesmo objeto, o que impede a divergencia que
o texto registra hoje entre a figura e as evidencias consolidadas.

In [ ]:
@secao("Figura 9")
def desenhar_figura_9() -> dict[str, float]:
    """Matriz de confusao da execucao promovida na particao de teste."""
    h = METRICAS[RUN_PRODUCAO]["holdout"]
    vn, fp, fn, vp = (int(h["true_negatives"]), int(h["false_positives"]),
                      int(h["false_negatives"]), int(h["true_positives"]))
    matriz = np.array([[vn, fp], [fn, vp]])

    fig, eixo = plt.subplots(figsize=(3.6, 3.1))
    eixo.imshow(matriz, cmap="Blues")
    eixo.grid(False)
    for i in range(2):
        for j in range(2):
            eixo.text(j, i, f"{matriz[i, j]}", ha="center", va="center", fontsize=13,
                      color="white" if matriz[i, j] > matriz.max() / 2 else "black")
    eixo.set_xticks([0, 1], ["Negativo predito", "Positivo predito"])
    eixo.set_yticks([0, 1], ["Negativo observado", "Positivo observado"])
    for lado in ("top", "right", "bottom", "left"):
        eixo.spines[lado].set_visible(False)
    fig.tight_layout()
    registrar_figura(
        fig, "figura_09_matriz_confusao",
        "Figura 9. Matriz de confusao da execucao de producao na particao de teste, "
        "com verdadeiros negativos, falsos positivos, falsos negativos e "
        "verdadeiros positivos",
    )
    plt.close(fig)

    VALORES.update({"vn": vn, "fp": fp, "fn": fn, "vp": vp,
                    "espec_teste": float(h["specificity"]),
                    "prec_teste": float(h["precision"]),
                    "positivos_teste": int(h["positives"])})
    negativos = vn + fp
    print(f"VN {vn}  FP {fp}  FN {fn}  VP {vp}   (n = {int(h['n'])})")
    print(f"Especificidade {fmt_br(h['specificity'])}: {fp} falsos positivos entre "
          f"{negativos} negativos, {fp / max(vp, 1):.2f} para cada verdadeiro positivo.")
    print(f"Precisao {fmt_br(h['precision'])}. Falsos negativos: {fn} de "
          f"{int(h['positives'])} positivos.")
    return h


metricas_producao = desenhar_figura_9()

## 7. Tabela 11, execucao de producao contra a regra clinica

Fonte: `evidencias/rule_baseline.json`, produzido por `src/models/rule_baseline.py`
sobre a mesma particao e a mesma funcao de metricas dos modelos. Os fatores da regra
saem da configuracao, nao de suposicao deste notebook.

In [ ]:
@secao("Tabela 11")
def montar_tabela_11() -> pd.DataFrame:
    """Confronta a execucao promovida com o piso clinico sem aprendizado."""
    regra = ler_json(EVIDENCIAS / "rule_baseline.json")
    teste = regra["sets"]["test"]
    h = METRICAS[RUN_PRODUCAO]["holdout"]

    print(f"Regra: positivo a partir de {regra['positive_from']} fatores, entre "
          f"{len(regra['factors'])} declarados:")
    for fator in regra["factors"]:
        print(f"  {fator['label_pt']}  ({fator['variable']} em "
              f"{', '.join(fator['categories'])})")

    chaves = [("auc_roc", "AUC-ROC"), ("sensitivity", "Sensibilidade"),
              ("specificity", "Especificidade"), ("precision", "Precisao"),
              ("f1", "F1"), ("accuracy", "Acuracia")]
    quadro = pd.DataFrame({
        "Metrica": [rotulo for _, rotulo in chaves],
        "Execucao de producao": [float(h[k]) for k, _ in chaves],
        "Regra clinica": [float(teste[k]) for k, _ in chaves],
    })
    quadro["Diferenca"] = quadro["Execucao de producao"] - quadro["Regra clinica"]
    VALORES["auc_regra"] = float(teste["auc_roc"])
    return quadro


tabela_11 = montar_tabela_11()
if tabela_11 is not None:
    registrar_tabela(
        "tabela_11", tabela_11,
        "Tabela 11. Execucao de producao e regra clinica na particao de teste",
        nota="a regra usa a fracao de fatores presentes como escore, o que permite "
             "calcular AUC; o rotulo vem do corte declarado em configuracao",
    )

## 8. Tabela 12, desempenho por subgrupo

Fonte: `evidencias/subgroup_metrics_lr_production.csv`, calculado sobre as predicoes
fora das dobras da execucao promovida, isto e, sobre o desenvolvimento completo sem
que nenhuma observacao tenha sido usada para ajustar o modelo que a pontuou.

In [ ]:
ROTULO_DIMENSAO: dict[str, str] = {"sex": "Sexo", "age_group": "Idade"}
ROTULO_GRUPO: dict[str, str] = {"80+": "80 ou mais"}


@secao("Tabela 12")
def montar_tabela_12() -> pd.DataFrame:
    """Desagrega o desempenho da execucao promovida por sexo e faixa etaria."""
    caminho = EVIDENCIAS / f"subgroup_metrics_{RUN_PRODUCAO}.csv"
    if not caminho.is_file():
        raise FileNotFoundError(f"artefato ausente: {caminho.relative_to(RAIZ)}")
    bruto = pd.read_csv(caminho)

    quadro = pd.DataFrame({
        "Dimensao e grupo": [
            f"{ROTULO_DIMENSAO.get(d, d)}, "
            f"{ROTULO_GRUPO.get(str(g), str(g)).replace('-', ' a ')}"
            for d, g in zip(bruto["dimension"], bruto["group"])
        ],
        "n": bruto["n"].astype(int),
        "Prevalencia": bruto["prevalence"].astype(float),
        "AUC-ROC": bruto["auc_roc"].astype(float),
        "Sensibilidade": bruto["sensitivity"].astype(float),
        "Especificidade": bruto["specificity"].astype(float),
    })
    minimo = ACEITACAO["subgroup_min_size"]
    pequenos = quadro[quadro["n"] < minimo]
    if not pequenos.empty:
        print(f"Subgrupos abaixo do minimo de {minimo}: "
              f"{', '.join(pequenos['Dimensao e grupo'])}")
    else:
        print(f"Nenhum subgrupo abaixo do minimo de {minimo}; "
              f"as {len(quadro)} linhas foram reportadas.")
    VALORES["n_subgrupos"] = len(quadro)
    VALORES["amplitude_sens_idade"] = (
        quadro[quadro["Dimensao e grupo"].str.startswith("Idade")]["Sensibilidade"]
        .agg(["min", "max"]).to_dict()
    )
    return quadro


tabela_12 = montar_tabela_12()
if tabela_12 is not None:
    registrar_tabela(
        "tabela_12", tabela_12,
        "Tabela 12. Desempenho da execucao de producao por subgrupo, fora das "
        "dobras de treino",
        casas={"n": 0},
    )

## 9. Figura 10, importancia por variavel bruta

Fonte: `evidencias/global_importance.json`, media do valor absoluto da contribuicao
linear no conjunto de treino, agregada da coluna binaria para a variavel bruta pelo
proprio pipeline (ADR 0003). Nao usa aproximacao amostral de valores de Shapley.

In [ ]:
@secao("Figura 10")
def desenhar_figura_10(topo: int = TOPO_IMPORTANCIA) -> pd.DataFrame:
    """Importancia das variaveis brutas de maior peso na execucao promovida."""
    dados = ler_json(EVIDENCIAS / "global_importance.json")
    quadro = pd.DataFrame(dados["raw_variables"]).sort_values(
        "importance", ascending=False).head(topo).reset_index(drop=True)

    # Rotulo em portugues, quando o mapa de interpretacao o fornece.
    interpretacao = ler_json(ARTEFATOS / RUN_PRODUCAO / "interpretation.json")
    rotulos: dict[str, str] = {}
    for item in interpretacao["importance"]:
        for bruta in item.get("source_raw", []):
            rotulos.setdefault(bruta, item["label_pt"])
    # Variaveis brutas distintas podem compartilhar o mesmo rotulo em portugues
    # (peso e altura alimentam a classificacao do IMC). Sem desambiguar, as barras
    # se sobrepoem no eixo categorico e a figura perde linhas em silencio.
    base = [rotulos.get(v, v) for v in quadro["variable"]]
    repetidos = {r for r in base if base.count(r) > 1}
    quadro["Variavel"] = [
        f"{rotulo} ({bruta})" if rotulo in repetidos else rotulo
        for rotulo, bruta in zip(base, quadro["variable"])
    ]
    if len(set(quadro["Variavel"])) != len(quadro):
        raise ValueError("rotulos ainda ambiguos apos a desambiguacao")

    fig, eixo = plt.subplots(figsize=(6.0, 3.6))
    eixo.barh(quadro["Variavel"][::-1], quadro["importance"][::-1], color="#3b6ea5")
    eixo.set_xlabel("Media do valor absoluto da contribuicao linear")
    eixo.grid(axis="y", visible=False)
    fig.tight_layout()
    registrar_figura(
        fig, "figura_10_importancia",
        f"Figura 10. Importancia das {topo} variaveis brutas de maior peso na "
        "execucao de producao, medida pela media do valor absoluto da contribuicao "
        "linear no conjunto de treino",
    )
    plt.close(fig)

    saida = quadro[["Variavel", "variable", "importance"]].rename(
        columns={"variable": "Nome no modelo", "importance": "Importancia"})
    VALORES["importancia_topo"] = saida.head(2).to_dict("records")
    return saida


importancias = desenhar_figura_10()
if importancias is not None:
    registrar_tabela("figura_10_dados", importancias,
                     "Dados de apoio da Figura 10", casas=3)

## 10. Tabela 13, fidelidade ao codigo de origem

Quatro colunas de valor, quatro origens:

| Coluna | Arquivo | O que e |
|---|---|---|
| Reexecucao da origem | `evidencias/fase0_reexecucao_metricas.json` | funcoes originais do notebook, executadas na Fase 0 |
| Fidelidade congelada | `evidencias/golden_metrics.json` | golden values, congelados em um commit anterior |
| Fidelidade atual | `artifacts/lr_fidelity/metrics.json` | execucao de fidelidade do pipeline de hoje |
| Producao | `artifacts/lr_production/metrics.json` | execucao promovida de hoje |

Tres distancias, porque cada par mede uma coisa diferente e **so vale comparar pares
que compartilham ambiente**:

| Distancia | Mede |
|---|---|
| Refatoracao | se a reescrita introduziu erro (origem contra fidelidade congelada) |
| Congelamento | a deriva entre o golden congelado e a execucao de fidelidade de hoje |
| Protocolo | o custo de ajustar o transformador so no treino (fidelidade atual contra producao) |

A distancia ao congelamento nao e, por si so, erro de refatoracao. O ADR 0014 documenta
que o `KNNImputer` do codigo original desempata distancias de forma dependente de
versao, o que reclassifica alguns registros na borda dos cortes de `pd.cut`; a tabela de
ambientes logo acima diz se essa e a explicacao no seu caso. Ela some quando o golden
for recongelado dentro da imagem de treino.

In [ ]:
# Caminho de cada metrica nos quatro arquivos, que usam vocabularios diferentes.
LINHAS_T13: list[tuple[str, str, str, str, float | None]] = [
    ("AUC-ROC, validacao cruzada", "cv10_LR.auc_mean", "cv.auc_roc_mean",
     "cv.summary.auc_roc_mean", FIDELIDADE["auc_tolerance"]),
    ("Sensibilidade, validacao cruzada", "cv10_LR.recall_mean", "cv.sensitivity_mean",
     "cv.summary.sensitivity_mean", FIDELIDADE["sensitivity_tolerance"]),
    ("AUC-ROC, particao de teste", "holdout_LR.auc", "holdout.auc_roc",
     "holdout.auc_roc", None),
    ("Sensibilidade, particao de teste", "holdout_LR.recall", "holdout.sensitivity",
     "holdout.sensitivity", None),
]


def caminho_aninhado(dados: Any, caminho: str) -> float:
    """Le um valor por caminho pontuado, devolvendo NaN quando ausente."""
    atual = dados
    for parte in caminho.split("."):
        if not isinstance(atual, dict) or parte not in atual:
            return float("nan")
        atual = atual[parte]
    return float(atual) if isinstance(atual, (int, float)) else float("nan")


def distancia(a: float, b: float) -> float:
    """Diferenca absoluta, ou NaN quando algum lado esta ausente."""
    return abs(a - b) if np.isfinite(a) and np.isfinite(b) else float("nan")


@secao("Ambientes dos artefatos de fidelidade")
def comparar_ambientes() -> pd.DataFrame:
    """Compara o ambiente de cada artefato de referencia com o do relatorio.

    O ADR 0014 declara que a referencia de fidelidade e ligada ao ambiente: o
    `KNNImputer` do codigo original desempata distancias de forma dependente de
    versao. Um artefato congelado fora do ambiente travado nao e comparavel com um
    gerado dentro dele, e essa checagem torna a diferenca visivel.
    """
    origem = ler_json(EVIDENCIAS / "fase0_reexecucao_metricas.json")
    pacotes = ("python", "pandas", "numpy", "scikit-learn")
    ambientes = {
        "Reexecucao da origem": origem.get("environment", {}),
        "Fidelidade congelada": GOLDEN.get("environment", {}),
        "Ambiente desta execucao do relatorio": {
            "python": PROVENIENCIA["python"],
            **{k: v for k, v in PROVENIENCIA["versoes_relatorio"].items() if k in pacotes},
        },
    }
    quadro = pd.DataFrame([
        {"Artefato": nome, **{p: amb.get(p, "[?]") for p in pacotes}}
        for nome, amb in ambientes.items()
    ])

    atual = ambientes["Ambiente desta execucao do relatorio"]
    congelado = ambientes["Fidelidade congelada"]
    divergem = [p for p in pacotes
                if congelado.get(p) and atual.get(p) and congelado[p] != atual[p]]
    VALORES["golden_no_ambiente_travado"] = not divergem
    if divergem:
        print(f"[REPRODUCIBILIDADE] O golden foi congelado com {', '.join(divergem)} "
              "diferentes do ambiente atual. Pelo ADR 0014, a distancia de ambiente "
              "abaixo e esperada e nao indica erro de refatoracao.")
        registrar_pendencia(
            "Golden values fora do ambiente travado",
            f"congelado com {', '.join(f'{p} {congelado[p]}' for p in divergem)}, "
            f"o relatorio roda com {', '.join(f'{p} {atual[p]}' for p in divergem)}",
            "make freeze-reference && python -m src.models.train --run lr_fidelity "
            "--freeze-golden, dentro da imagem de treino (ADR 0014)",
        )
    else:
        print("Golden congelado no mesmo ambiente do relatorio.")
    print("\nRessalva: `artifacts/<run>/metrics.json` nao registra o ambiente que o "
          "produziu, entao a linha do relatorio so equivale ao das execucoes atuais "
          "quando o notebook roda na mesma imagem que treinou. Registrar o ambiente "
          "em `_persist_artifacts` tornaria isso auditavel em vez de suposto.")
    return quadro


ambientes = comparar_ambientes()
if ambientes is not None:
    registrar_tabela("tabela_13_ambientes", ambientes,
                     "Tabela de apoio. Ambiente de cada artefato de referencia",
                     nota="ADR 0014: a referencia de fidelidade e ligada ao ambiente")


@secao("Tabela 13")
def montar_tabela_13() -> pd.DataFrame:
    """Confronta origem, fidelidade congelada, fidelidade atual e producao."""
    origem = ler_json(EVIDENCIAS / "fase0_reexecucao_metricas.json")
    if RUN_FIDELIDADE not in METRICAS:
        raise KeyError(f"metrics.json de {RUN_FIDELIDADE} ausente")
    fidelidade_atual = METRICAS[RUN_FIDELIDADE]
    producao = METRICAS[RUN_PRODUCAO]

    linhas: list[dict[str, Any]] = []
    for rotulo, via_origem, via_golden, via_run, tolerancia in LINHAS_T13:
        v_origem = caminho_aninhado(origem, via_origem)
        v_congelada = caminho_aninhado(GOLDEN, via_golden)
        v_atual = caminho_aninhado(fidelidade_atual, via_run)
        v_producao = caminho_aninhado(producao, via_run)

        d_refatoracao = distancia(v_origem, v_congelada)   # mesmo ambiente
        d_congelamento = distancia(v_congelada, v_atual)   # ver ADR 0014
        d_protocolo = distancia(v_atual, v_producao)       # mesmo ambiente

        if tolerancia is None:
            situacao = "nao declarada"
        else:
            excedeu = [d for d in (d_refatoracao, d_protocolo)
                       if np.isfinite(d) and d > tolerancia]
            situacao = "FORA DA TOLERANCIA" if excedeu else "dentro"
        linhas.append({
            "Metrica": rotulo,
            "Reexecucao da origem": v_origem,
            "Fidelidade congelada": v_congelada,
            "Fidelidade atual": v_atual,
            "Producao": v_producao,
            "Dif. refatoracao": d_refatoracao,
            "Dif. congelamento": d_congelamento,
            "Dif. protocolo": d_protocolo,
            "Tolerancia": fmt_br(tolerancia, 3) if tolerancia is not None
            else "nao declarada",
            "Situacao": situacao,
        })

    quadro = pd.DataFrame(linhas)
    fora = quadro.loc[quadro["Situacao"] == "FORA DA TOLERANCIA", "Metrica"].tolist()
    VALORES["fidelidade_ok"] = not fora
    if fora:
        print(f"[DADOS] Fora da tolerancia: {', '.join(fora)}.")
    else:
        print("Refatoracao e protocolo dentro das tolerancias declaradas.")

    maior_congelamento = quadro["Dif. congelamento"].max()
    VALORES["maior_dif_congelamento"] = float(maior_congelamento)
    if np.isfinite(maior_congelamento) and maior_congelamento > 0:
        causa = ("deriva de ambiente prevista pelo ADR 0014"
                 if not VALORES.get("golden_no_ambiente_travado", True)
                 else "mudanca de codigo ou de ambiente desde o congelamento")
        print(f"Maior distancia ao congelamento: {fmt_br(maior_congelamento, 6)}. "
              f"Causa provavel: {causa}. Some quando o golden for recongelado com "
              "--freeze-golden na imagem de treino.")
    if (quadro["Dif. refatoracao"].fillna(1) == 0).all():
        print("A reescrita reproduziu a origem sem diferenca nas casas registradas, "
              "metrica a metrica.")
    return quadro


tabela_13 = montar_tabela_13()
if tabela_13 is not None:
    registrar_tabela(
        "tabela_13", tabela_13,
        "Tabela 13. Fidelidade ao codigo de origem, por execucao de referencia",
        casas=6,
        nota="refatoracao compara origem e fidelidade congelada, ambas no mesmo "
             "ambiente; protocolo compara fidelidade atual e producao, ambas no "
             "ambiente de hoje; congelamento e a deriva entre as duas epocas. "
             f"Commit {commit_curto()}; tolerancias de configs/config.yaml",
    )

## 11. Tabela 14, teste de carga e eq. (9)

Fonte: `evidencias/load_test_*.json`. A premissa de preco horario vem de
`configs/config.yaml`, secao de carga, e alimenta a eq. (9) do custo por mil
requisicoes.

In [ ]:
@secao("Tabela 14")
def montar_tabela_14() -> pd.DataFrame:
    """Monta a tabela de carga a partir de todos os resultados persistidos."""
    # Glob sem o sublinhado: o nome padrao do script e "load_test", sem sufixo, e
    # "load_test_*.json" deixaria esse arquivo de fora em silencio.
    arquivos = sorted(EVIDENCIAS.glob("load_test*.json"))
    if not arquivos:
        raise FileNotFoundError("nenhum load_test*.json em evidencias/")

    portao = ACEITACAO["max_p95_latency_ms"]
    linhas: list[dict[str, Any]] = []
    for caminho in arquivos:
        d = ler_json(caminho)
        latencia = d.get("latency_ms", {})
        alvo = ("Conteiner da API" if "stub" not in caminho.stem
                else "Servico equivalente")
        linhas.append({
            "Execucao": alvo,
            "Clientes": int(d.get("concurrency", 0)),
            "p50": float(latencia.get("p50", np.nan)),
            "p95": float(latencia.get("p95", np.nan)),
            "p99": float(latencia.get("p99", np.nan)),
            "Vazao": float(d.get("throughput_rps", np.nan)),
            "Requisicoes": int(d.get("requests", 0)),
            "CPU media %": float(d.get("resources", {}).get("cpu_percent_mean", np.nan)),
            f"Portao {portao} ms": "aprovado" if float(
                latencia.get("p95", np.inf)) <= portao else "reprovado",
        })
    quadro = pd.DataFrame(linhas).sort_values(["Execucao", "Clientes"]).reset_index(drop=True)

    # A condicao declarada e a concorrencia 4. Entre as execucoes que a atendem, a
    # do conteiner tem precedencia: e a medicao que a secao anterior declara. O
    # servico equivalente so entra enquanto o conteiner nao tiver sido medido.
    alvo = CONFIG["load_test"]["concurrency"]
    candidatas = quadro[quadro["Clientes"] == alvo]
    for execucao in ("Conteiner da API", "Servico equivalente"):
        declarada = candidatas[candidatas["Execucao"] == execucao]
        if declarada.empty:
            continue
        VALORES["p95_declarado"] = float(declarada["p95"].iloc[0])
        VALORES["vazao_declarada"] = float(declarada["Vazao"].iloc[0])
        VALORES["requisicoes_declaradas"] = int(declarada["Requisicoes"].iloc[0])
        VALORES["origem_carga"] = execucao
        print(f"Condicao declarada ({alvo} clientes) lida de: {execucao}")
        break
    if not (quadro["Execucao"] == "Conteiner da API").any():
        registrar_pendencia(
            "Tabela 14, linha do conteiner da API",
            "nenhum resultado de carga contra o conteiner",
            "docker compose -f docker/docker-compose.yml up -d api && "
            "python scripts/load_test.py --url http://localhost:8000 "
            "--concurrency 4 --requests 1000",
        )
    return quadro


tabela_14 = montar_tabela_14()
if tabela_14 is not None:
    registrar_tabela(
        "tabela_14", tabela_14,
        "Tabela 14. Teste de carga por concorrencia, sobre um processo de trabalho",
        casas={"p50": 2, "p95": 2, "p99": 2, "Vazao": 2, "CPU media %": 1,
               "Clientes": 0, "Requisicoes": 0},
        nota="as execucoes de servico equivalente usam o mesmo servico de predicao, "
             "o mesmo contrato e a mesma camada intermediaria da API",
    )

    carga_cfg = CONFIG.get("load_test", CONFIG.get("monitoring", {}).get("load_test", {}))
    preco = carga_cfg.get("cost_per_hour")
    moeda = carga_cfg.get("cost_currency", "USD")
    if preco is None:
        for bloco in CONFIG.values():
            if isinstance(bloco, dict) and "cost_per_hour" in bloco:
                preco, moeda = bloco["cost_per_hour"], bloco.get("cost_currency", "USD")
                break
    vazao = VALORES.get("vazao_declarada")
    if preco is None:
        registrar_pendencia("Equacao (9)", "cost_per_hour ausente de configs/config.yaml",
                            "Declarar a premissa de preco horario na configuracao.")
    elif vazao:
        custo = (float(preco) * 1000) / (3600 * vazao)
        VALORES["custo_por_mil"] = custo
        VALORES["preco_hora"] = float(preco)
        VALORES["moeda"] = moeda
        print(f"\nEquacao (9): C = ({fmt_br(preco, 2)} x 1.000) / (3.600 x "
              f"{fmt_br(vazao, 2)}) = {fmt_br(custo, 4)} {moeda} por mil requisicoes")
        print(f"Premissa: {carga_cfg.get('cost_basis', '')}".strip())

## 12. Tabela 15, desvio de dados

Fonte: `evidencias/drift_simulated.json` e `drift_baseline.json`. A tabela reporta
as variaveis em alerta do cenario simulado e, do cenario de linha de base, o maior
PSI e o menor valor-p observados, que sao os numeros que o texto cita.

In [ ]:
@secao("Tabela 15")
def montar_tabela_15() -> pd.DataFrame:
    """Reune as variaveis que decidiram cada cenario de desvio."""
    simulado = ler_json(EVIDENCIAS / "drift_simulated.json")
    baseline = ler_json(EVIDENCIAS / "drift_baseline.json")
    limiares = simulado["thresholds"]

    def variaveis(payload: dict[str, Any]) -> pd.DataFrame:
        return pd.DataFrame(payload["offline"]["variables"])

    sim = variaveis(simulado)
    em_alerta = sim[sim["alert"]].sort_values("psi", ascending=False)

    linhas = [{
        "Cenario e variavel": f"Simulado, {linha['name']}",
        "PSI": float(linha["psi"]),
        "KS": float(linha["ks_statistic"]) if pd.notna(linha["ks_statistic"]) else np.nan,
        "Valor-p": float(linha["ks_pvalue"]) if pd.notna(linha["ks_pvalue"]) else np.nan,
        "Situacao": "alerta",
    } for _, linha in em_alerta.iterrows()]

    base = variaveis(baseline)
    maior_psi = base.loc[base["psi"].idxmax()]
    com_p = base[base["ks_pvalue"].notna()]
    menor_p = com_p.loc[com_p["ks_pvalue"].idxmin()] if not com_p.empty else None
    linhas.append({
        "Cenario e variavel": f"Linha de base, maior PSI ({maior_psi['name']})",
        "PSI": float(maior_psi["psi"]),
        "KS": float(maior_psi["ks_statistic"]) if pd.notna(maior_psi["ks_statistic"]) else np.nan,
        "Valor-p": float(menor_p["ks_pvalue"]) if menor_p is not None else np.nan,
        "Situacao": "sem alerta e sem aviso",
    })

    VALORES.update({
        "alertas_simulado": int(sim["alert"].sum()),
        "alertas_baseline": int(base["alert"].sum()),
        "avisos_baseline": len(baseline["offline"]["warnings"]),
        "maior_psi_baseline": float(maior_psi["psi"]),
        "menor_psi_alerta": float(em_alerta["psi"].min()) if not em_alerta.empty else np.nan,
        "psi_alerta_limiar": limiares["psi_alert"],
        "psi_aviso_limiar": limiares["psi_warning"],
        "min_variaveis_retreino": CONFIG["monitoring"]["retraining"]["min_variables_in_alert"],
    })
    folga = VALORES["menor_psi_alerta"] / max(VALORES["maior_psi_baseline"], 1e-12)
    print(f"Simulado: {VALORES['alertas_simulado']} variaveis em alerta de "
          f"{len(sim)} avaliadas. Linha de base: {VALORES['alertas_baseline']} alertas "
          f"e {VALORES['avisos_baseline']} avisos.")
    print(f"Folga entre cenarios: {folga:.1f} vezes "
          f"({fmt_br(VALORES['maior_psi_baseline'])} contra "
          f"{fmt_br(VALORES['menor_psi_alerta'])}).")
    print(f"Sinalizador de retreino dispara com {VALORES['min_variaveis_retreino']} "
          f"variaveis; o cenario simulado produziu {VALORES['alertas_simulado']}.")
    return pd.DataFrame(linhas)


tabela_15 = montar_tabela_15()
if tabela_15 is not None:
    registrar_tabela(
        "tabela_15", tabela_15,
        "Tabela 15. Desvio de dados nos cenarios de linha de base e simulado",
        nota="a estatistica KS nao se aplica a variavel categorica; a ultima linha "
             "reune o maior PSI e o menor valor-p do cenario de linha de base",
    )

## 13. Rastreabilidade, reprodutibilidade e testes

Os numeros que o texto usa no subtopico de engenharia, lidos dos artefatos:
o registro do modelo (dado da Figura 11), a verificacao de reprodutibilidade e a
cobertura de testes.

In [ ]:
@secao("Rastreabilidade e reprodutibilidade")
def montar_engenharia() -> pd.DataFrame:
    """Reune registro do modelo, reprodutibilidade e cobertura em uma tabela."""
    linhas: list[dict[str, Any]] = []

    if REGISTRO:
        linhas += [
            {"Item": "Modelo registrado", "Valor": REGISTRO.get("name", "?")},
            {"Item": "Versao em producao", "Valor": REGISTRO.get("version", "?")},
            {"Item": "Estagio", "Valor": REGISTRO.get("stage", "?")},
            {"Item": "Alias", "Valor": REGISTRO.get("alias", "?")},
            {"Item": "Execucao promovida", "Valor": REGISTRO.get("run", "?")},
            {"Item": "Identificador da execucao", "Valor": REGISTRO.get("run_id", "?")},
            {"Item": "Resumo do dado bruto",
             "Valor": str(REGISTRO.get("raw_dataset_sha256", "?"))[:16] + "..."},
        ]
        VALORES["versao_modelo"] = REGISTRO.get("version")
    else:
        registrar_pendencia("Registro do modelo", "registration.json ausente",
                            "python -m src.models.train --all")

    caminho_rep = EVIDENCIAS / "reproducibility_two_runs.json"
    if caminho_rep.is_file():
        rep = ler_json(caminho_rep)
        linhas += [
            {"Item": "Execucoes repetidas comparadas", "Valor": len(rep.get("runs", []))},
            {"Item": "Resumos de artefato identicos",
             "Valor": fmt_br(rep.get("artifact_hashes_identical"))},
            {"Item": "Metricas identicas",
             "Valor": fmt_br(rep.get("all_metrics_identical"))},
            {"Item": "Ambiente da verificacao", "Valor": rep.get("environment", "?")},
        ]
        VALORES["reprodutibilidade"] = bool(
            rep.get("artifact_hashes_identical") and rep.get("all_metrics_identical"))
        VALORES["runs_reprodutibilidade"] = len(rep.get("runs", []))
    else:
        registrar_pendencia("Reprodutibilidade", "reproducibility_two_runs.json ausente",
                            "python scripts/run_pipeline.py --no-docker, duas vezes")

    # Dois formatos convivem: a tabela em canos do rastreamento proprio e a linha
    # TOTAL do pytest-cov. Um extrator por expressao regular cobre os dois sem supor
    # separador, e ignora o arquivo quando a linha TOTAL nao traz percentual.
    import re as _re

    def percentual_total(texto: str) -> float | None:
        """Extrai o percentual da linha TOTAL de um relatorio de cobertura."""
        for linha in reversed(texto.splitlines()):
            if not linha.strip().upper().startswith("TOTAL"):
                continue
            achados = _re.findall(r"(\d+(?:[.,]\d+)?)\s*%", linha)
            if achados:
                return float(achados[-1].replace(",", "."))
        return None

    cobertura = None
    # Ordem de preferencia: relatorio de ferramenta antes do rastreamento proprio.
    for nome in ("coverage_summary.txt", "coverage.txt", "coverage_report.txt",
                 "coverage_local_trace.txt"):
        caminho = EVIDENCIAS / nome
        if not caminho.is_file():
            continue
        valor = percentual_total(caminho.read_text(encoding="utf-8", errors="replace"))
        if valor is None:
            print(f"  {nome}: sem linha TOTAL com percentual, ignorado")
            continue
        cobertura = valor
        linhas.append({"Item": f"Cobertura de testes ({nome})",
                       "Valor": f"{fmt_br(cobertura, 1)} %"})
        VALORES["cobertura"] = cobertura
        VALORES["cobertura_oficial"] = "trace" not in nome
        break
    if cobertura is None:
        registrar_pendencia("Cobertura de testes", "nenhum relatorio de cobertura legivel",
                            "pytest --cov=src --cov-report=term --cov-report=xml")
    elif not VALORES.get("cobertura_oficial"):
        registrar_pendencia(
            "Cobertura oficial",
            "valor medido por rastreamento de execucao, nao por ferramenta de cobertura",
            "pytest --cov=src --cov-report=term > evidencias/coverage_summary.txt",
        )

    linhas.append({"Item": "Etapas encadeadas por comando unico",
                   "Valor": "scripts/run_pipeline.py"
                   if (RAIZ / "scripts" / "run_pipeline.py").is_file() else "[AUSENTE]"})
    VALORES["comando_unico"] = (RAIZ / "scripts" / "run_pipeline.py").is_file()
    return pd.DataFrame(linhas)


engenharia = montar_engenharia()
if engenharia is not None:
    registrar_tabela("tabela_engenharia", engenharia,
                     "Tabela de apoio. Rastreabilidade, reprodutibilidade e testes")

## 14. Tabela 16, confronto dos objetivos especificos

Construida de forma declarativa: o limiar vem de `configs/config.yaml`, o valor
observado sai dos objetos ja calculados neste notebook e a situacao e derivada da
comparacao. A tabela nao envelhece em silencio quando uma metrica muda.

In [ ]:
# Volume declarado do teste de carga: vem da configuracao.
VOLUME_DECLARADO: int = CONFIG["load_test"]["requests"]

# Vazao minima: o Projeto de Pesquisa declara 50 req/s, mas o valor nao esta em
# configs/config.yaml. Fica explicito aqui e vira pendencia, porque criterio de
# aceitacao fora da configuracao e numero magico.
MIN_VAZAO_RPS: float = float(
    ACEITACAO.get("min_throughput_rps", 50)
)
if "min_throughput_rps" not in ACEITACAO:
    registrar_pendencia(
        "Limiar de vazao fora da configuracao",
        "acceptance.min_throughput_rps nao existe em configs/config.yaml; "
        f"o relatorio usa {MIN_VAZAO_RPS} req/s, declarado no Projeto de Pesquisa",
        "Acrescentar 'min_throughput_rps: 50' na secao acceptance de configs/config.yaml",
    )


def obs(chave: str) -> float:
    """Le um valor calculado, devolvendo NaN quando ausente."""
    valor = VALORES.get(chave)
    return float(valor) if isinstance(valor, (int, float)) and not isinstance(valor, bool) \
        else float("nan")


CRITERIOS: list[dict[str, Any]] = [
    {"oe": "OE1", "criterio": "Etapas encadeadas por comando unico",
     "limiar": "seis etapas",
     "observado": lambda: "seis etapas" if VALORES.get("comando_unico") else "[PEND.]",
     "situacao": lambda: "atendido" if VALORES.get("comando_unico") else "[PENDENTE]"},
    {"oe": "OE2", "criterio": "Execucoes registradas com versionamento",
     "limiar": f"{len(ORDEM_RUNS)} execucoes",
     "observado": lambda: f"{VALORES.get('n_execucoes', 0)} execucoes",
     "situacao": lambda: "atendido"
     if VALORES.get("n_execucoes", 0) >= len(ORDEM_RUNS) else "reprovado"},
    {"oe": "OE3", "criterio": "Cobertura de testes", "limiar": "80 %",
     "observado": lambda: f"{fmt_br(obs('cobertura'), 1)} %",
     "situacao": lambda: "[PENDENTE]" if not np.isfinite(obs("cobertura"))
     else ("atendido" if obs("cobertura") >= 80 and VALORES.get("cobertura_oficial")
           else "reprovado" if obs("cobertura") < 80 else "[PENDENTE]")},
    {"oe": "OE4", "criterio": "Latencia no percentil 95",
     "limiar": f"{ACEITACAO['slo_p95_latency_ms']} ms",
     "observado": lambda: f"{fmt_br(obs('p95_declarado'), 2)} ms",
     "situacao": lambda: "[PENDENTE]" if not np.isfinite(obs("p95_declarado"))
     else ("atendido" if obs("p95_declarado") <= ACEITACAO["slo_p95_latency_ms"]
           else "reprovado")},
    {"oe": "OE5", "criterio": "AUC-ROC na validacao cruzada",
     "limiar": fmt_br(ACEITACAO["min_auc"], 2),
     "observado": lambda: fmt_br(obs("auc_cv_producao"), 4),
     "situacao": lambda: "[PENDENTE]" if not np.isfinite(obs("auc_cv_producao"))
     else ("atendido" if obs("auc_cv_producao") >= ACEITACAO["min_auc"] else "reprovado")},
    {"oe": "OE5", "criterio": "Sensibilidade na validacao cruzada",
     "limiar": fmt_br(ACEITACAO["min_sensitivity"], 2),
     "observado": lambda: fmt_br(obs("sens_cv_producao"), 4),
     "situacao": lambda: "[PENDENTE]" if not np.isfinite(obs("sens_cv_producao"))
     else ("atendido" if obs("sens_cv_producao") >= ACEITACAO["min_sensitivity"]
           else "reprovado")},
    # O limiar de vazao nao consta de configs/config.yaml: vem do Projeto de Pesquisa.
    # Declarado aqui e sinalizado abaixo, para nao virar numero magico silencioso.
    {"oe": "OE5", "criterio": "Vazao sob carga",
     "limiar": f"{MIN_VAZAO_RPS:g} req/s",
     "observado": lambda: f"{fmt_br(obs('vazao_declarada'), 1)} req/s",
     "situacao": lambda: "[PENDENTE]" if not np.isfinite(obs("vazao_declarada"))
     else ("atendido" if obs("vazao_declarada") >= MIN_VAZAO_RPS else "reprovado")},
    {"oe": "OE5", "criterio": "Volume do teste de carga",
     "limiar": f"{VOLUME_DECLARADO} requisicoes",
     "observado": lambda: f"{VALORES.get('requisicoes_declaradas', '?')} requisicoes",
     "situacao": lambda: "[PENDENTE]"
     if VALORES.get("requisicoes_declaradas") is None
     else ("atendido"
           if VALORES["requisicoes_declaradas"] >= VOLUME_DECLARADO else "reprovado")},
    {"oe": "OE5", "criterio": "Desvio ausente na linha de base",
     "limiar": "nenhuma variavel",
     "observado": lambda: f"{VALORES.get('alertas_baseline', '?')} variaveis",
     "situacao": lambda: "atendido" if VALORES.get("alertas_baseline") == 0
     else "reprovado"},
    {"oe": "OE5", "criterio": "Desvio detectado no cenario simulado",
     "limiar": "uma ou mais",
     "observado": lambda: f"{VALORES.get('alertas_simulado', '?')} variaveis",
     "situacao": lambda: "atendido" if VALORES.get("alertas_simulado", 0) >= 1
     else "reprovado"},
    {"oe": "OE5", "criterio": "Alerta por queda de AUC-ROC acima de 5 %",
     "limiar": "regra ativa", "observado": lambda: "sem rotulo em producao",
     "situacao": lambda: "nao verificavel"},
    {"oe": "OE6", "criterio": "Execucoes repetidas com resumos identicos",
     "limiar": "duas execucoes",
     "observado": lambda: "duas execucoes" if VALORES.get("reprodutibilidade")
     else "[PEND.]",
     "situacao": lambda: "atendido" if VALORES.get("reprodutibilidade") else "[PENDENTE]"},
    {"oe": "OE6", "criterio": "Fidelidade dentro da tolerancia",
     "limiar": f"AUC {FIDELIDADE['auc_tolerance']}, sens. "
               f"{FIDELIDADE['sensitivity_tolerance']}",
     "observado": lambda: "dentro" if VALORES.get("fidelidade_ok") else "fora",
     "situacao": lambda: "atendido" if VALORES.get("fidelidade_ok") else "reprovado"},
    {"oe": "OE4", "criterio": "Disponibilidade em homologacao", "limiar": "99 %",
     "observado": lambda: "ambiente ausente", "situacao": lambda: "nao verificavel"},
]


@secao("Tabela 16")
def montar_tabela_16() -> pd.DataFrame:
    """Confronta cada criterio declarado com o valor observado nesta execucao."""
    linhas = [{
        "OE": item["oe"], "Criterio": item["criterio"], "Limiar": item["limiar"],
        "Observado": item["observado"](), "Situacao": item["situacao"](),
    } for item in CRITERIOS]
    return pd.DataFrame(linhas).sort_values("OE", kind="stable").reset_index(drop=True)


tabela_16 = montar_tabela_16()
if tabela_16 is not None:
    registrar_tabela(
        "tabela_16", tabela_16,
        "Tabela 16. Confronto dos objetivos especificos com os limiares declarados",
        nota="limiares de configs/config.yaml, secoes acceptance e fidelity",
    )
    resumo = tabela_16["Situacao"].value_counts()
    print("\nResumo dos criterios:")
    print(resumo.to_string())
    reprovados = tabela_16[tabela_16["Situacao"] == "reprovado"]
    for _, linha in reprovados.iterrows():
        registrar_pendencia(f"{linha['OE']}: {linha['Criterio']}",
                            f"observado {linha['Observado']} contra limiar {linha['Limiar']}",
                            "Ver a secao correspondente para o comando.")

## 15. Divergencias entre o texto da secao e os artefatos atuais

Os artefatos foram regerados depois que a versao v3 do texto foi escrita, e alguns
numeros mudaram. Esta secao confronta o que o texto afirma hoje com o que os
artefatos dizem agora, para que a revisao seja mecanica em vez de conferencia
manual celula a celula.

A lista de valores do texto esta declarada abaixo e precisa ser atualizada quando o
texto for revisado.

In [ ]:
# Valores como aparecem na versao v3 do texto. Fonte: TCC_Resultados_e_Discussao_v3.
# Atualize esta lista quando o texto for revisado, para que a checagem continue util.
TEXTO_V3: list[tuple[str, float, Callable[[], float]]] = [
    ("Tabela 10, lr_production, AUC VC", 0.7484, lambda: cv(RUN_PRODUCAO, "auc_roc_mean")),
    ("Tabela 10, lr_production, Sens. VC", 0.6794,
     lambda: cv(RUN_PRODUCAO, "sensitivity_mean")),
    ("Tabela 10, lr_production, Espec. VC", 0.6885,
     lambda: cv(RUN_PRODUCAO, "specificity_mean")),
    ("Tabela 10, lr_production, AUC teste", 0.7468, lambda: holdout(RUN_PRODUCAO, "auc_roc")),
    ("Tabela 10, lr_fidelity, AUC VC", 0.7488, lambda: cv(RUN_FIDELIDADE, "auc_roc_mean")),
    ("Tabela 10, lr_fidelity, Sens. VC", 0.6813,
     lambda: cv(RUN_FIDELIDADE, "sensitivity_mean")),
    ("Tabela 11, regra clinica, AUC-ROC", 0.6764, lambda: obs("auc_regra")),
    ("Figura 9, verdadeiros negativos", 727.0, lambda: obs("vn")),
    ("Figura 9, falsos positivos", 303.0, lambda: obs("fp")),
    ("Figura 9, falsos negativos", 68.0, lambda: obs("fn")),
    ("Figura 9, verdadeiros positivos", 143.0, lambda: obs("vp")),
    ("Texto, especificidade no teste", 0.7058, lambda: obs("espec_teste")),
    ("Texto, precisao no teste", 0.3206, lambda: obs("prec_teste")),
    ("Texto, p95 na condicao declarada (ms)", 148.87, lambda: obs("p95_declarado")),
    ("Texto, vazao (req/s)", 30.3, lambda: obs("vazao_declarada")),
    ("Texto, maior PSI na linha de base", 0.0325, lambda: obs("maior_psi_baseline")),
    ("Texto, menor PSI em alerta", 0.7899, lambda: obs("menor_psi_alerta")),
    ("Texto, cobertura (%)", 79.9, lambda: obs("cobertura")),
]

# Tolerancia de comparacao: o texto arredonda a quatro casas, logo meia unidade da
# ultima casa e diferenca de arredondamento, nao divergencia de conteudo.
TOLERANCIA_TEXTO: float = 5e-5


@secao("Divergencias com o texto")
def montar_divergencias() -> pd.DataFrame:
    """Confronta os valores do texto v3 com os artefatos atuais."""
    linhas: list[dict[str, Any]] = []
    for rotulo, no_texto, extrator in TEXTO_V3:
        atual = extrator()
        if not np.isfinite(atual):
            situacao, diferenca = "[PEND.]", float("nan")
        else:
            escala = max(abs(no_texto), 1.0)
            diferenca = atual - no_texto
            relativo = abs(diferenca) / escala
            situacao = ("igual" if abs(diferenca) <= TOLERANCIA_TEXTO * escala
                        else "arredondamento" if relativo < 5e-4
                        else "ATUALIZAR TEXTO")
        linhas.append({"Item": rotulo, "No texto v3": no_texto,
                       "No artefato atual": atual, "Diferenca": diferenca,
                       "Situacao": situacao})
    quadro = pd.DataFrame(linhas)
    atualizar = quadro[quadro["Situacao"] == "ATUALIZAR TEXTO"]
    if not atualizar.empty:
        print(f"{len(atualizar)} valores do texto precisam ser atualizados:")
        for _, linha in atualizar.iterrows():
            print(f"  {linha['Item']}: {fmt_br(linha['No texto v3'])} -> "
                  f"{fmt_br(linha['No artefato atual'])}")
        registrar_pendencia(
            "Texto da secao de Resultados",
            f"{len(atualizar)} valores divergem dos artefatos atuais",
            "Substituir pelos valores da coluna 'No artefato atual' desta tabela.",
        )
    else:
        print("Nenhuma divergencia relevante entre o texto v3 e os artefatos atuais.")
    return quadro


divergencias = montar_divergencias()
if divergencias is not None:
    registrar_tabela(
        "divergencias_texto", divergencias,
        "Tabela de controle. Valores do texto v3 contra os artefatos atuais",
        casas=4,
        nota="nao integra o TCC; serve a revisao do texto antes do deposito",
    )

## 16. Consolidacao e pendencias

Reune tudo em um arquivo unico, na ordem da secao de Resultados, e emite o relatorio
de pendencias que substitui a lista de fechamento mantida a mao no documento.

In [ ]:
ORDEM_ENTREGA: list[tuple[str, str]] = [
    ("tabela_10", "Tabela 10"),
    ("tabela_10_estendida", "Tabela 10 estendida"),
    ("tabela_coeficientes", "Coeficientes, eq. (8)"),
    ("figura_08_roc", "Figura 8"),
    ("figura_09_matriz_confusao", "Figura 9"),
    ("tabela_11", "Tabela 11"),
    ("tabela_12", "Tabela 12"),
    ("figura_10_importancia", "Figura 10"),
    ("figura_10_dados", "Dados da Figura 10"),
    ("tabela_13", "Tabela 13"),
    ("tabela_engenharia", "Rastreabilidade e reprodutibilidade"),
    ("tabela_14", "Tabela 14"),
    ("tabela_15", "Tabela 15"),
    ("tabela_16", "Tabela 16"),
    ("divergencias_texto", "Controle de divergencias"),
]

partes: list[str] = [
    "# Resultados regerados",
    "",
    f"Gerado em {PROVENIENCIA['gerado_em_utc']} (UTC), commit {commit_curto()}, "
    f"tag {PROVENIENCIA['git_tag']}, "
    f"arvore {'limpa' if PROVENIENCIA['git_limpo'] else 'SUJA'}.",
    "",
    f"Execucao promovida: `{RUN_PRODUCAO}`, versao "
    f"{REGISTRO.get('version', '?')} em {REGISTRO.get('stage', '?')}. "
    f"Semente {SEED}, k = {N_FOLDS}, limiar {THRESHOLD}.",
    "",
    "Proveniencia completa em `proveniencia.json`.",
    "",
]

for nome, rotulo in ORDEM_ENTREGA:
    md = SAIDA / "tabelas" / f"{nome}.md"
    png = SAIDA / "figuras" / f"{nome}.png"
    legenda = SAIDA / "figuras" / f"{nome}.txt"
    if md.exists():
        partes += [md.read_text(encoding="utf-8"), ""]
    elif png.exists():
        partes += [legenda.read_text(encoding="utf-8") if legenda.exists() else rotulo,
                   "", f"![{rotulo}](figuras/{nome}.png)", "",
                   "Fonte: Resultados originais da pesquisa", ""]
    else:
        partes += [f"## {rotulo}", "", "[PENDENTE] nao gerado nesta execucao.", ""]

(SAIDA / "Resultados_Regerados.md").write_text("\n".join(partes), encoding="utf-8")

# Pendencias que nenhum artefato resolve: dependem de servico no ar ou de fonte externa.
PENDENCIAS_FIXAS: list[dict[str, str]] = [
    {"item": "Figura 11, registro de modelos",
     "motivo": "captura de tela de servico no ar",
     "acao": "docker compose -f docker/docker-compose.yml up -d mlflow, abrir a aba "
             "Models e capturar em evidencias/mlflow_model_registry.png"},
    {"item": "Figura 12, pagina do modelo na interface",
     "motivo": "captura de tela de servico no ar",
     "acao": "docker compose -f docker/docker-compose.yml up -d api app e capturar em "
             "evidencias/painel_streamlit_modelo.png, tarjando credencial"},
    {"item": "Figura 13, painel de observabilidade",
     "motivo": "captura de tela de servico no ar",
     "acao": "docker compose -f docker/docker-compose.yml up -d prometheus grafana, "
             "gerar trafego e capturar em evidencias/dashboard_grafana.png"},
    {"item": "Paginas da dissertacao de origem",
     "motivo": "quatro citacoes com [PENDENTE] no lugar da pagina",
     "acao": "Preencher a partir do exemplar consultado de Dias (2024)"},
]

relatorio = pd.DataFrame(PENDENCIAS + PENDENCIAS_FIXAS)
relatorio.to_csv(SAIDA / "pendencias.csv", index=False, encoding="utf-8")
(SAIDA / "PENDENCIAS.md").write_text(
    "\n".join(["# Pendencias apos a regeracao", "",
               f"Execucao de {PROVENIENCIA['gerado_em_utc']} (UTC).", "",
               quadro_para_markdown(relatorio) if not relatorio.empty
               else "Nenhuma pendencia registrada."]) + "\n",
    encoding="utf-8")


def exibir_markdown(texto: str) -> None:
    """Renderiza Markdown na celula, com queda para texto puro fora do Jupyter."""
    try:
        from IPython.display import Markdown
        display(Markdown(texto))
    except ImportError:
        print(texto)


print(f"Entrega consolidada em: {SAIDA}\n")
indice = pd.DataFrame(
    [{"arquivo": str(c.relative_to(SAIDA)), "KB": round(c.stat().st_size / 1024, 1)}
     for c in sorted(SAIDA.rglob("*")) if c.is_file()])
display(indice)

print(f"\n{len(relatorio)} pendencias registradas:\n")
exibir_markdown(quadro_para_markdown(relatorio) if not relatorio.empty
                else "Nenhuma pendencia registrada.")
exibir_markdown("\n".join(partes))